# 02 — Preprocessing và Gate A

Notebook tích hợp các module đã kiểm thử độc lập. Dùng lại student split của EDA.
Không fit model; không tính AUC/Brier/Log Loss hoặc phân phối nhãn test.
Các file parquet chứa nhãn phục vụ thí nghiệm sau được lưu cục bộ và Git-ignore.


In [1]:
from pathlib import Path
import sys, json, subprocess
ROOT = Path.cwd()
if not (ROOT / 'scripts/prepare_data.py').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
assert (ROOT / 'scripts/prepare_data.py').exists()
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
                        cwd=ROOT, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, 'Gate A tests failed; do not prepare/train'



test_each_students_labels_cannot_influence_own_oof_features (test_gate_a_independent.IndependentDifficultyTests) ... ok
test_heldout_labels_cannot_change_any_difficulty (test_gate_a_independent.IndependentDifficultyTests) ... ok
test_unseen_problem_fallback_is_fold_local_not_global (test_gate_a_independent.IndependentDifficultyTests) ... ok
test_every_current_and_future_label_cannot_change_available_history (test_gate_a_independent.IndependentHistoryTests) ... ok
test_manual_interleaved_skill_history_and_cleaned_warmup (test_gate_a_independent.IndependentHistoryTests) ... ok
test_split_rejects_overlap_and_unassigned_students (test_gate_a_independent.IndependentHistoryTests) ... ok
test_declared_global_parameter_mutation_is_rejected (test_gate_a_independent.IndependentSequentialTests) ... ok
test_predict_before_observe_reset_and_matching_masks_on_shuffled_input (test_gate_a_independent.IndependentSequentialTests) ... ok
test_clean_rejects_invalid_targets_identity_and_duplicate_order (t

## Dựng dữ liệu
OOF train chia theo student; validation/test chỉ nhận thống kê full-train. Feature lịch sử chỉ dùng tương tác trước đó.

In [2]:
from scripts.prepare_data import prepare
summary = prepare()


{
  "status": "data_prepared_not_trained",
  "raw_sha256": "162ef8d2d28bcbfea6591a282994062bd8d5eaa00636544292a0d268dca6e5da",
  "split_manifest_sha256": "7b7a5e10f9c8b6dcd8ef4efa3965c3ef738ff32d23907e91e0f2050622afd27d",
  "raw_rows": 346860,
  "clean_rows": 236068,
  "warmup": 5,
  "oof_folds": 5,
  "smoothing_alpha": 10.0,
  "config_version": "rq1-a-v1",
  "config_sha256": "60001cd0375c8e10ee4a09464a6dd9892935cf6d89790de74bd313c1e7e20a01",
  "label_dependent_test_statistics_reported": false,
  "partitions": {
    "train": {
      "rows": 168431,
      "students": 2802,
      "scored_rows": 155418,
      "students_without_scored_rows": 492,
      "sha256": "4168f4c556b4eb94d4838847861bada648522874af3da383d058d53dfe62acaf"
    },
    "validation": {
      "rows": 35135,
      "students": 600,
      "scored_rows": 32382,
      "students_without_scored_rows": 119,
      "sha256": "ae0ba35895e959157afcdd7931b9e96e15f62a49bc7f19eccf472b179399bd87"
    },
    "test": {
      "rows": 32502,

## Kiểm tra đối chiếu evaluator
Dùng model giả cố định 0.5 để kiểm tra giao thức trên vài trajectory train; không phải baseline nghiên cứu.

In [3]:
import pandas as pd
from src.evaluation.sequential import sequential_predict
class ProtocolProbe:
    def reset(self, user_id): self.seen = 0
    def predict(self, row):
        assert 'correct' not in row
        assert 'hint_count' not in row
        return 0.5
    def update(self, row): self.seen += 1
train = pd.read_parquet(ROOT / 'data/processed/train.parquet')
users = sorted(train.user_id.unique())[:5]
sample = train[train.user_id.isin(users)].copy()
result = sequential_predict(sample, ProtocolProbe(), warmup=json.loads((ROOT / 'configs/protocol_a.json').read_text(encoding='utf-8'))['evaluation']['warmup'])
expected = sample.set_index('source_row').scored
actual = result.set_index('source_row').is_scored
assert actual.eq(expected.reindex(actual.index)).all()
print('Evaluator mask matches preprocessing:', len(result), 'train rows')
print('Gate A integration checks completed; no model training performed.')


Evaluator mask matches preprocessing: 266 train rows
Gate A integration checks completed; no model training performed.
